In [10]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

import numpy as np
import bilby.gw.conversion
import pycbc
import pycbc.psd
from scipy.optimize import differential_evolution
from pycbc.waveform import get_td_waveform
from pycbc.types import TimeSeries, FrequencySeries
from gwtorch.modules.gw_utils import compute_lensed_waveform

In [2]:
DELTA_T = 1.0 / 4096
F_LOWER = 5.0

In [3]:
def objective_function(params_array, args):
    chirp_mass, mass_ratio, spin1z, spin2z, coa_phase = params_array
    eccentric_signal, F_LOW, DELTA_T, distance  = args
    try:
        mass1, mass2 = bilby.gw.conversion.chirp_mass_and_mass_ratio_to_component_masses(chirp_mass=chirp_mass, mass_ratio=mass_ratio)

        template_hp, template_hc = get_td_waveform(approximant='teobresums', mass1=mass1, mass2=mass2,
                                                    lambda1=0, lambda2=0,
                                                    spin1z=spin1z, spin2z=spin2z,
                                                    distance=distance, delta_t=DELTA_T,
                                                    ecc=0.0, coa_phase=coa_phase, f_lower=F_LOW)
        
        ecc_copy = eccentric_signal.copy()
        temp_copy = template_hp.copy()

        target_len = max(len(ecc_copy), len(temp_copy))
        ecc_copy.resize(target_len)
        temp_copy.resize(target_len)

        duration = temp_copy.duration
        flen     = target_len // 2 + 1
        psd = pycbc.psd.EinsteinTelescopeP1600143(flen, 1/duration, F_LOWER)
        
        match_value, _ = pycbc.filter.match(ecc_copy, temp_copy, psd=psd, low_frequency_cutoff=F_LOW)

        return 1.0 - match_value
    except Exception as e:
        print(f"Error in objective function: {e}")
        return 1.0

In [ ]:
bounds = [
    (10, 100), # Chirp Mass
    (0.1, 1), # Mass Ratio
    (-0.9, 0.9), # spin1z
    (-0.9, 0.9), # spin2z
    (0, 2*np.pi), # coa_phase
]

### Eccentric Waveform

In [20]:
chirp_mass_, mass_ratio_ = 20, 0.5

mass1_, mass2_ = bilby.gw.conversion.chirp_mass_and_mass_ratio_to_component_masses(chirp_mass=chirp_mass_, mass_ratio=mass_ratio_)
spin1z_, spin2z_ = 0.1, -0.3
coa_phase_ = 3.14

ecc_signal, _ = get_td_waveform(approximant='teobresums', mass1=mass1_, mass2=mass2_,
                                 lambda1=0, lambda2=0,
                                 spin1z=spin1z_, spin2z=spin2z_,
                                 distance=500, delta_t=DELTA_T,
                                 ecc=0.3, coa_phase=coa_phase_, f_lower=F_LOWER)

In [21]:
args_for_optimizer = (ecc_signal, F_LOWER, DELTA_T, 500)

In [22]:
result = differential_evolution(
    func=objective_function,
    bounds=bounds,
    args=(args_for_optimizer,),
    strategy='best1bin',
    maxiter=200,
    popsize=20,
    disp=True,
    workers=-1,
    updating="deferred"
)

differential_evolution step 1: f(x)= 0.6341657767343285
differential_evolution step 2: f(x)= 0.4912285083735821
differential_evolution step 3: f(x)= 0.4912285083735821
differential_evolution step 4: f(x)= 0.4912285083735821
differential_evolution step 5: f(x)= 0.4912285083735821
differential_evolution step 6: f(x)= 0.4535451427453958
differential_evolution step 7: f(x)= 0.44583727124250894
differential_evolution step 8: f(x)= 0.44583727124250894
differential_evolution step 9: f(x)= 0.44479082710475326
differential_evolution step 10: f(x)= 0.4318059682210752
differential_evolution step 11: f(x)= 0.4318059682210752
differential_evolution step 12: f(x)= 0.4318059682210752
differential_evolution step 13: f(x)= 0.4318059682210752
differential_evolution step 14: f(x)= 0.4318059682210752
differential_evolution step 15: f(x)= 0.4318059682210752
differential_evolution step 16: f(x)= 0.4318059682210752
differential_evolution step 17: f(x)= 0.42858088232057634
differential_evolution step 18: f(x)

In [23]:
best_fit_params = result.x
fitting_factor = 1.0 - result.fun

In [31]:
print("\n" + "="*60)
print("Global Optimization Complete!")
print(f"Fitting Factor: {fitting_factor:.6f}")
print("="*60)
print(f"{'Parameter':<17} | {'True Value':<15} | {'Best-Fit Quasi-Circular Value':<20}")
print("-"*60)
print(f"{'Chirp Mass':<17} | {chirp_mass_:<15.4f} | {best_fit_params[0]:<20.4f}")
print(f"{'Mass Ratio':<17} | {mass_ratio_:<15.4f} | {best_fit_params[1]:<20.4f}")
print(f"{'Spin1z':<17} | {spin1z_:<15.4f} | {best_fit_params[2]:<20.4f}")
print(f"{'Spin2z':<17} | {spin2z_:<15.4f} | {best_fit_params[3]:<20.4f}")
print(f"{'Coalescence Phase':<15} | {coa_phase_:<15.4f} | {best_fit_params[4]:<20.4f}")
print("="*60)


Global Optimization Complete!
Fitting Factor: 0.574359
Parameter         | True Value      | Best-Fit Quasi-Circular Value
------------------------------------------------------------
Chirp Mass        | 20.0000         | 20.9898             
Mass Ratio        | 0.5000          | 0.7668              
Spin1z            | 0.1000          | -0.3105             
Spin2z            | -0.3000         | 0.6622              
Coalescence Phase | 3.1400          | 0.5331              


### Lens Waveform

In [32]:
chirp_mass_, mass_ratio_ = 20, 0.5

mass1_, mass2_ = bilby.gw.conversion.chirp_mass_and_mass_ratio_to_component_masses(chirp_mass=chirp_mass_, mass_ratio=mass_ratio_)
spin1z_, spin2z_ = 0.1, -0.3
coa_phase_ = 3.14

m_lens, y_lens = 1e4, 0.3

hp, hc = get_td_waveform(approximant='teobresums', mass1=mass1_, mass2=mass2_,
                                 lambda1=0, lambda2=0,
                                 spin1z=spin1z_, spin2z=spin2z_,
                                 distance=500, delta_t=DELTA_T,
                                 ecc=0.0, coa_phase=coa_phase_, f_lower=F_LOWER)

sp_lensed, sc_lensed, t_delay = compute_lensed_waveform(hp, hc, m_lens, y_lens)

In [33]:
args_for_optimizer = (sp_lensed, F_LOWER, DELTA_T, 500)

In [34]:
result = differential_evolution(
    func=objective_function,
    bounds=bounds,
    args=(args_for_optimizer,),
    strategy='best1bin',
    maxiter=200,
    popsize=20,
    disp=True,
    workers=-1,
    updating="deferred"
)

differential_evolution step 1: f(x)= 0.5664263125881721
differential_evolution step 2: f(x)= 0.5664263125881721
differential_evolution step 3: f(x)= 0.5664263125881721
differential_evolution step 4: f(x)= 0.28858862170142374
differential_evolution step 5: f(x)= 0.28858862170142374
differential_evolution step 6: f(x)= 0.28858862170142374
differential_evolution step 7: f(x)= 0.28564286831940866
differential_evolution step 8: f(x)= 0.28564286831940866
differential_evolution step 9: f(x)= 0.28564286831940866
differential_evolution step 10: f(x)= 0.28564286831940866
differential_evolution step 11: f(x)= 0.28564286831940866
differential_evolution step 12: f(x)= 0.28564286831940866
differential_evolution step 13: f(x)= 0.28564286831940866
differential_evolution step 14: f(x)= 0.28564286831940866
differential_evolution step 15: f(x)= 0.23228734483639546
differential_evolution step 16: f(x)= 0.23228734483639546
differential_evolution step 17: f(x)= 0.23228734483639546
differential_evolution ste

In [35]:
best_fit_params = result.x
fitting_factor = 1.0 - result.fun

In [36]:
print("\n" + "="*60)
print("Global Optimization Complete!")
print(f"Fitting Factor: {fitting_factor:.6f}")
print("="*60)
print(f"{'Parameter':<17} | {'True Value':<15} | {'Best-Fit Quasi-Circular Value':<20}")
print("-"*60)
print(f"{'Chirp Mass':<17} | {chirp_mass_:<15.4f} | {best_fit_params[0]:<20.4f}")
print(f"{'Mass Ratio':<17} | {mass_ratio_:<15.4f} | {best_fit_params[1]:<20.4f}")
print(f"{'Spin1z':<17} | {spin1z_:<15.4f} | {best_fit_params[2]:<20.4f}")
print(f"{'Spin2z':<17} | {spin2z_:<15.4f} | {best_fit_params[3]:<20.4f}")
print(f"{'Coalescence Phase':<15} | {coa_phase_:<15.4f} | {best_fit_params[4]:<20.4f}")
print("="*60)


Global Optimization Complete!
Fitting Factor: 0.795506
Parameter         | True Value      | Best-Fit Quasi-Circular Value
------------------------------------------------------------
Chirp Mass        | 20.0000         | 19.9723             
Mass Ratio        | 0.5000          | 0.8433              
Spin1z            | 0.1000          | -0.0763             
Spin2z            | -0.3000         | -0.1860             
Coalescence Phase | 3.1400          | 3.8775              
